In [2]:
from langchain_community.document_loaders import DataFrameLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

C:\Users\mdzak\AppData\Local\Temp\ipykernel_21896\1879199775.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DataFrameLoader
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import pandas as pd

pd.set_option('display.max_colwidth', 100)

In [5]:
books = pd.read_csv("books_cleaned.csv")

In [6]:
books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_description,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gb...,"A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an ...",2004.0,3.85,247.0,361.0,0,Gilead,"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade,..."
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gb...,A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed play by Charles...,2000.0,3.83,241.0,5164.0,0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed p...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2TgANz74C&printsec=frontcover&img=1&zoom=1&source=gb...,"A memorable, mesmerizing heroine Jennifer -- brilliant, beautiful, an attorney on the way up unt...",1993.0,3.93,512.0,29532.0,0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine Jennifer -- brilliant, beautiful, an attorney on ..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5XsFcpGIC&printsec=frontcover&img=1&zoom=1&source=gb...,"Lewis' work on the nature of love divides love into four categories; Affection, Friendship, Eros...",2002.0,4.15,170.0,33684.0,0,The Four Loves,"9780006280897 Lewis' work on the nature of love divides love into four categories; Affection, Fr..."
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uVe5QK-gC&printsec=frontcover&img=1&zoom=1&source=gb...,"""In The Problem of Pain, C.S. Lewis, one of the most renowned Christian authors and thinkers, ex...",2002.0,4.09,176.0,37569.0,0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Lewis, one of the most renowned Christian authors an..."


In [7]:
loader = DataFrameLoader(books, page_content_column="tagged_description")
documents = loader.load()

In [8]:
len(documents)

5693

In [9]:
documents[0]

Document(metadata={'isbn13': 9780002005883, 'isbn10': '0002005883', 'title': 'Gilead', 'authors': 'Marilynne Robinson', 'categories': 'Fiction', 'thumbnail': 'http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'description': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best fri

The `CharacterTextSplitter` was raising a `ValueError` because recent versions of `langchain` require `chunk_size` to be a positive integer.

A more direct and efficient approach is to load the data directly from the pandas DataFrame using `DataFrameLoader`. This avoids the intermediate step of saving to a text file and then splitting it, making the code cleaner and avoiding the error.

In [10]:
db_books = Chroma.from_documents(
    documents,
    embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
)

In [11]:
# import google.generativeai as genai
#
# print("Available embedding models:")
# for m in genai.list_models():
#   if 'embedContent' in m.supported_generation_methods:
#     print(m.name)

In [12]:
query = "A book to teach children about nature"

docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='23d5b49d-ff6a-4335-aeb6-224cdc16ec3f', metadata={'isbn10': '006757520X', 'categories': 'Nature', 'average_rating': 4.39, 'missing_description': 0, 'thumbnail': 'http://books.google.com/books/content?id=Zee5SAOqO2UC&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'title_and_subtitle': 'The Sense of Wonder', 'title': 'The Sense of Wonder', 'isbn13': 9780067575208, 'num_pages': 112.0, 'published_year': 1998.0, 'authors': 'Rachel Carson', 'ratings_count': 1160.0, 'description': 'First published more than three decades ago, this reissue of Rachel Carson\'s award-winning classic brings her unique vision to a new generation of readers. Stunning new photographs by Nick Kelsh beautifully complement Carson\'s intimate account of adventures with her young nephew, Roger, as they enjoy walks along the rocky coast of Maine and through dense forests and open fields, observing wildlife, strange plants, moonlight and storm clouds, and listening to the "living music" of insects in the un

- We're getting correct books based on description. However, we want to show users details like title, author etc, instead of long descriptions.

In [13]:
books[books["isbn13"]== int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_description,title_and_subtitle,tagged_description
462,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5SAOqO2UC&printsec=frontcover&img=1&zoom=1&source=gb...,"First published more than three decades ago, this reissue of Rachel Carson's award-winning class...",1998.0,4.39,112.0,1160.0,0,The Sense of Wonder,"9780067575208 First published more than three decades ago, this reissue of Rachel Carson's award..."


In [15]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int =10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [16]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_description,title_and_subtitle,tagged_description
107,9780060192501,006019250X,The Illustrated Alchemist,Paulo Coelho;Alan R. Clarke;Moebius,Fiction,http://books.google.com/books/content?id=1YX2usKBAMIC&printsec=frontcover&img=1&zoom=1&source=gb...,"This fable aims teaches the reader to open their mind, listen to their heart and most importantl...",1998.0,3.85,198.0,236.0,0,The Illustrated Alchemist: A Fable About Following Your Dream,"9780060192501 This fable aims teaches the reader to open their mind, listen to their heart and m..."
150,9780060546571,0060546573,Three Rotten Eggs,Gregory Maguire,Juvenile Fiction,http://books.google.com/books/content?id=t2pWlCpY3jQC&printsec=frontcover&img=1&zoom=1&source=gb...,The students of Miss Earth's class in rural Vermont experience an eventful spring when they beco...,2005.0,3.74,240.0,76.0,0,Three Rotten Eggs,9780060546571 The students of Miss Earth's class in rural Vermont experience an eventful spring ...
227,9780060765453,0060765453,The Chronicles of Narnia Movie Tie-in Edition (adult),C. S. Lewis,Fiction,http://books.google.com/books/content?id=4qVxvQEACAAJ&printsec=frontcover&img=1&zoom=1&source=gb...,"When Digory and Polly try to return the wicked witch Jadis to her own world, the magic gets mixe...",2005.0,4.26,766.0,1099.0,0,The Chronicles of Narnia Movie Tie-in Edition (adult),"9780060765453 When Digory and Polly try to return the wicked witch Jadis to her own world, the m..."
235,9780060782139,0060782137,Time For Kids: Butterflies!,Editors of TIME For Kids,Juvenile Nonfiction,http://books.google.com/books/content?id=OdZxnI53jj4C&printsec=frontcover&img=1&zoom=1&source=gb...,"Butterflies There are 20,000 different kinds of butterflies in the world. Many have brightly col...",2006.0,4.00,32.0,20.0,0,Time For Kids: Butterflies!,"9780060782139 Butterflies There are 20,000 different kinds of butterflies in the world. Many hav..."
277,9780060885373,0060885378,Little House in the Big Woods,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=7JctNysS3xkC&printsec=frontcover&img=1&zoom=1&source=gb...,"A year in the life of two young girls growing up on the Wisconsin frontier, as they help their m...",2007.0,4.18,198.0,193862.0,0,Little House in the Big Woods,"9780060885373 A year in the life of two young girls growing up on the Wisconsin frontier, as the..."
449,9780064434980,0064434982,The Deer in the Wood,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=V7YDWiHI7m4C&printsec=frontcover&img=1&zoom=1&source=gb...,"Even the youngest child can enjoy a special adaptation of a classic Little House tale, as Laura ...",1999.0,4.17,32.0,302.0,0,The Deer in the Wood,9780064434980 Even the youngest child can enjoy a special adaptation of a classic Little House t...
462,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5SAOqO2UC&printsec=frontcover&img=1&zoom=1&source=gb...,"First published more than three decades ago, this reissue of Rachel Carson's award-winning class...",1998.0,4.39,112.0,1160.0,0,The Sense of Wonder,"9780067575208 First published more than three decades ago, this reissue of Rachel Carson's award..."
907,9780143037392,0143037390,The Read-aloud Handbook,Jim Trelease,Language Arts & Disciplines,http://books.google.com/books/content?id=B2_yUfmc6dsC&printsec=frontcover&img=1&zoom=1&source=gb...,"Explains the importance of reading aloud to children, offers guidance on how to set up a read-al...",2006.0,4.40,432.0,4122.0,0,The Read-aloud Handbook,"9780143037392 Explains the importance of reading aloud to children, offers guidance on how to se..."
1151,9780241003008,0241003008,The Very Hungry Caterpillar,Eric Carle,Babytime resource,http://books.google.com/books/content?id=DpGEQgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gb...,Eric Carle's 

---